In [ ]:
!pip install -q -U transformers accelerate bitsandbytes
!pip install -U bitsandbytes>=0.46.1

In [ ]:
!nvidia-smi

In [ ]:
import os
import json
import re
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


INPUT_FILE = "/content/retrieval_baseline_final.json"
OUTPUT_FILE = "/content/relevance_evaluation_final.json"


MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

model.eval()

print("Model loaded successfully.")

In [ ]:
INPUT_FILE = "/content/retrieval_baseline_final.json"
OUTPUT_FILE = "/content/relevance_evaluation_final.json"

In [ ]:
SYSTEM_PROMPT = """
You are an expert evaluator of information retrieval
for a medical question-answering system.

Your task is to evaluate how relevant each retrieved chunk
is for answering the given multiple-choice medical question.

IMPORTANT:
- Do not provide any reasoning or explanation.
- Do not output <think> or </think>.
- Do not analyze your decision in natural language.
- Return ONLY the JSON object requested below.

Evaluate each chunk independently.

4 = The chunk contains sufficient information to directly
    support the correct answer.

3 = The chunk is strongly relevant and contains evidence
    supporting the correct answer, but additional information
    may be needed.

2 = The chunk is related to the topic but does not provide
    meaningful evidence for the correct answer.

1 = The chunk has only weak or tangential relevance.

0 = The chunk is irrelevant.

Return exactly one score for every retrieved chunk.

Return ONLY valid JSON:

{
  "evaluations": [
    {
      "rank": 1,
      "relevance_score": 4
    }
  ]
}
"""



def format_options(options):
    lines = []

    for letter in ["a", "b", "c", "d", "e"]:
        if letter in options:
            lines.append(
                f"{letter}. {options[letter]}"
            )

    return "\n".join(lines)


def format_chunks(retrieved):

    blocks = []

    for item in retrieved:

        block = (
            f"--- Chunk Rank {item['rank']} ---\n"
            f"{item['text']}"
        )

        blocks.append(block)

    return "\n\n".join(blocks)




def build_prompt(item):

    options_text = format_options(
        item["options"]
    )

    chunks_text = format_chunks(
        item.get("retrieved", [])
    )

    prompt = f"""
Question:
{item["question"]}

Options:
{options_text}

Correct answer:
{item["answer"]}

Reference answer:
{item.get("answer_text", "")}

Reference explanation:
{item.get("explanation", "")}

Retrieved chunks:

{chunks_text}

Evaluate every retrieved chunk independently.

Return ONLY valid JSON in this format:

{{
  "evaluations": [
    {{
      "rank": 1,
      "relevance_score": 4
    }}
  ]
}}
"""

    return prompt




def generate_judgment(prompt):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False
        )

    generated_tokens = outputs[
        0
    ][
        inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()



def parse_json_response(response):

    response = response.strip()

    # Remove markdown fences
    response = re.sub(
        r"```json\s*",
        "",
        response,
        flags=re.IGNORECASE
    )

    response = re.sub(
        r"```\s*",
        "",
        response
    )

    response = response.strip()
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        pass
    match = re.search(
        r"\{.*\}",
        response,
        flags=re.DOTALL
    )

    if match:

        json_text = match.group(0)

        try:
            return json.loads(json_text)

        except json.JSONDecodeError:
            pass

    raise ValueError(
        f"Could not parse model response as JSON:\n{response}"
    )


def validate_evaluation(
    evaluation,
    retrieved
):

    if "evaluations" not in evaluation:
        raise ValueError(
            "Missing 'evaluations' field."
        )

    evaluations = evaluation["evaluations"]

    expected_ranks = {
        item["rank"]
        for item in retrieved
    }

    returned_ranks = {
        item.get("rank")
        for item in evaluations
    }

    if expected_ranks != returned_ranks:
        raise ValueError(
            f"Rank mismatch. "
            f"Expected {expected_ranks}, "
            f"got {returned_ranks}"
        )

    for item in evaluations:

        score = item.get(
            "relevance_score"
        )

        if not isinstance(score, int):
            raise ValueError(
                f"Invalid score: {score}"
            )

        if score < 0 or score > 4:
            raise ValueError(
                f"Score must be 0-4, got {score}"
            )

    return True

In [ ]:
with open(
    INPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    dataset = json.load(f)

print(
    f"Total questions: {len(dataset)}"
)

print(
    f"Total retrieved chunks: "
    f"{sum(len(x.get('retrieved', [])) for x in dataset)}"
)


if os.path.exists(OUTPUT_FILE):

    with open(
        OUTPUT_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        final_results = json.load(f)

else:

    final_results = []


processed_ids = {
    item["question_id"]
    for item in final_results
    if "question_id" in item
}

print(
    f"Already processed: "
    f"{len(processed_ids)}"
)

for index, item in enumerate(
    dataset,
    start=1
):

    question_id = item["id"]
    if question_id in processed_ids:

        print(
            f"[{index}/{len(dataset)}] "
            f"{question_id} - SKIPPED"
        )

        continue

    print(
        f"\n[{index}/{len(dataset)}] "
        f"{question_id}"
    )

    retrieved = item.get(
        "retrieved",
        []
    )

    if not retrieved:

        print(
            "No retrieved chunks."
        )

        final_results.append({

            "question_id":
                question_id,

            "chapter":
                item.get(
                    "chapter",
                    ""
                ),

            "question_number":
                item.get(
                    "question_number",
                    ""
                ),

            "question":
                item["question"],

            "correct_answer":
                item["answer"],

            "error":
                "No retrieved chunks."

        })

        continue

    prompt = build_prompt(item)
    try:

        raw_response = generate_judgment(
            prompt
        )

        evaluation = parse_json_response(
            raw_response
        )

        validate_evaluation(
            evaluation,
            retrieved
        )
        result = {

            "question_id":
                question_id,

            "chapter":
                item.get(
                    "chapter",
                    ""
                ),

            "question_number":
                item.get(
                    "question_number",
                    ""
                ),

            "question":
                item["question"],

            "correct_answer":
                item["answer"],

            "answer_text":
                item.get(
                    "answer_text",
                    ""
                ),

            "explanation":
                item.get(
                    "explanation",
                    ""
                ),

            "evaluations":
                evaluation["evaluations"]

        }

        final_results.append(
            result
        )

        print(
            "Evaluation completed."
        )
        with open(
            OUTPUT_FILE,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                final_results,
                f,
                ensure_ascii=False,
                indent=2
            )

        print(
            f"Checkpoint saved "
            f"({len(final_results)} questions)"
        )

    except Exception as e:

        print(
            f"ERROR: {e}"
        )

        final_results.append({

            "question_id":
                question_id,

            "chapter":
                item.get(
                    "chapter",
                    ""
                ),

            "question_number":
                item.get(
                    "question_number",
                    ""
                ),

            "question":
                item["question"],

            "correct_answer":
                item["answer"],

            "error":
                str(e)

        })
        with open(
            OUTPUT_FILE,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                final_results,
                f,
                ensure_ascii=False,
                indent=2
            )

        continue

Total questions: 232
Total retrieved chunks: 2320
Already processed: 0

[1/232] CAMPBELL_0001
✓ Evaluation completed.
✓ Checkpoint saved (1 questions)

[2/232] CAMPBELL_0002
✓ Evaluation completed.
✓ Checkpoint saved (2 questions)

[3/232] CAMPBELL_0003
✓ Evaluation completed.
✓ Checkpoint saved (3 questions)

[4/232] CAMPBELL_0004
✗ ERROR: CUDA out of memory. Tried to allocate 62.50 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.77 GiB is free. Including non-PyTorch memory, this process has 12.79 GiB memory in use. Of the allocated memory 12.00 GiB is allocated by PyTorch, and 681.14 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

[5/232] CAMPBELL_0005
✓ Evaluation completed.
✓ Checkpoint saved (5 questio